# DAY 3 — Full Session Content
### Antrix Computer Academy — 30-Day Machine Learning Course (Pure ML Track)

---

## Session Title
**Exploratory Data Analysis, Data Cleaning & Feature Engineering Intro — Mini-Project 1 Kickoff**

## Duration
1.5–2 hours (plus ongoing project work time outside class)

## Recap of Day 4 (2 minutes, before diving in)
Quick verbal check-in — cold-call 2–3 students:
1. Why can two datasets have identical summary statistics but look completely different when plotted?
2. What does the box in a boxplot represent? What do the dots beyond the whiskers represent?
3. Why is spotting skew/outliers visually useful before training a model?

## Learning Objectives
By the end of today, students will be able to:
1. Run a structured Exploratory Data Analysis (EDA) process on a new, unseen dataset
2. Detect and handle missing data, duplicates, and outliers using both code and the visual techniques from Day 4
3. Apply introductory feature engineering: encoding categorical variables, scaling numeric variables, and creating simple derived features
4. Combine everything from Module 1 (stats, viz, cleaning) into a single, coherent EDA report
5. Understand the Mini-Project 1 brief, choose a dataset, and begin work

## Session Agenda (Time-boxed)

| Time | Segment |
|---|---|
| 0:00–0:05 | Day 4 recap (quick quiz) |
| 0:05–0:15 | Why EDA Ties Module 1 Together |
| 0:15–0:35 | Handling Missing Data |
| 0:35–0:45 | Handling Duplicates |
| 0:45–1:00 | Handling Outliers |
| 1:00–1:20 | Feature Engineering Intro (encoding, scaling, simple new features) |
| 1:20–1:35 | The Full EDA Workflow, End to End |
| 1:35–1:50 | Mini-Project 1 Brief — dataset, deliverables, rubric |
| 1:50–2:00 | Recap + project work time begins |

---

## 1. Why EDA Ties Module 1 Together (10 minutes)

**Framing:**
> Every technique from the last four days exists to answer one practical question: *"Is this dataset ready to train a model on, and if not, what do I need to fix?"* EDA is where statistics, math intuition, and visualization come together into a repeatable process.

**The EDA loop, put on the board:**
```
1. Look at the shape & structure   → .shape, .info(), .head()
2. Check for missing data          → .isnull().sum()
3. Check for duplicates            → .duplicated().sum()
4. Look at distributions           → histograms, boxplots (Day 4)
5. Look at relationships           → scatterplots, correlation heatmap (Day 2, Day 4)
6. Spot and handle outliers        → boxplots, IQR / z-score
7. Engineer/transform features     → encoding, scaling, new features
8. Document what you found         → this becomes your EDA report
```

> Tell students: *"You will run this exact loop on almost every new dataset for the rest of your ML career. Today, you make it a habit."*

---

## 2. Handling Missing Data (20 minutes, hands-on)

### 2.1 Detecting Missing Data

```python
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [25, np.nan, 35, 40, np.nan],
    "salary": [50000, 60000, np.nan, 80000, 55000],
    "department": ["Sales", "HR", "Sales", np.nan, "Engineering"]
})

print(df.isnull().sum())          # count of missing values per column
print(df.isnull().mean() * 100)   # percentage missing per column
```

### 2.2 Strategy 1 — Drop Missing Data

```python
# Drop rows with ANY missing value
df_dropped_rows = df.dropna()

# Drop columns with too many missing values (e.g., >50%)
df_dropped_cols = df.dropna(axis=1, thresh=len(df) * 0.5)
```

> **When to use:** only when missing data is a small fraction of rows, or a column is so sparse it isn't salvageable. Dropping rows carelessly can throw away a lot of otherwise-good data.

### 2.3 Strategy 2 — Imputation (Filling In Values)

```python
# Numeric columns — fill with mean or median
df["age"] = df["age"].fillna(df["age"].median())
df["salary"] = df["salary"].fillna(df["salary"].mean())

# Categorical columns — fill with mode (most frequent value)
df["department"] = df["department"].fillna(df["department"].mode()[0])

print(df)
```

**Discussion prompt (ties back to Day 2):** *"Given what we know about mean vs. median and outliers, when would you choose median imputation over mean imputation for a numeric column?"* (Answer: when the column is skewed or has outliers — the mean gets dragged by them, the median doesn't.)

### 2.4 Strategy 3 — Forward/Backward Fill (useful for ordered/time-series data)

```python
df_ordered = pd.DataFrame({"reading": [10, np.nan, np.nan, 13, 15]})

df_ordered["filled_forward"] = df_ordered["reading"].fillna(method="ffill")
df_ordered["filled_backward"] = df_ordered["reading"].fillna(method="bfill")

print(df_ordered)
```

---

## 3. Handling Duplicates (10 minutes)

```python
df_dupes = pd.DataFrame({
    "name": ["Aditi", "John", "Aditi", "Wei"],
    "score": [90, 85, 90, 78]
})

print(df_dupes.duplicated().sum())    # count of exact duplicate rows

df_clean = df_dupes.drop_duplicates()
print(df_clean)

# Duplicates based on a subset of columns (e.g., same name, different score = still worth investigating)
print(df_dupes.duplicated(subset=["name"]).sum())
```

**Discussion prompt:** *"If two rows have the same customer ID but different purchase amounts, is that a 'duplicate' to drop, or two separate real transactions? How would you tell the difference?"*

---

## 4. Handling Outliers (15 minutes)

### 4.1 The IQR Method (connects directly to Day 4's boxplot)

```python
data = pd.Series([45, 48, 50, 52, 55, 51, 49, 300])

Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data < lower_bound) | (data > upper_bound)]
print("Outliers:", outliers.tolist())
```

> This is the exact same 1.5×IQR rule that determines where a boxplot's whiskers end (Day 4) — now expressed in code instead of just read off a chart.

### 4.2 The Z-Score Method (connects to Day 2's standard deviation)

```python
mean = data.mean()
std = data.std()

z_scores = (data - mean) / std
outliers_z = data[abs(z_scores) > 3]     # more than 3 std devs from the mean
print("Outliers (z-score):", outliers_z.tolist())
```

### 4.3 What to Do With Outliers — Options, Not a Single Rule

| Option | When to use |
|---|---|
| **Remove** | Outlier is clearly a data-entry error (e.g., age = 999) |
| **Cap/Clip** | Outlier is real but extreme — cap at a reasonable max/min instead of deleting |
| **Keep, but note it** | Outlier is a genuine rare event that matters for the problem (e.g., fraud detection — the outliers ARE what you're looking for!) |
| **Transform** | Apply a log transform to reduce the outlier's influence without removing it (see Section 5) |

> Important nuance to state explicitly: *"Not every outlier is 'bad data.' Whether to remove one depends entirely on the problem you're solving — this is a judgment call, not a formula."*

---

## 5. Feature Engineering Intro (20 minutes)

> Today is intentionally "just enough to prep data for Mini-Project 1." A full deep dive on encoding strategies, scaling techniques, and feature importance comes back on Day 22 — don't over-teach it now.

### 5.1 Encoding Categorical Variables

```python
df_cat = pd.DataFrame({"department": ["Sales", "HR", "Engineering", "Sales"]})

# One-hot encoding — creates a binary column per category
df_encoded = pd.get_dummies(df_cat, columns=["department"])
print(df_encoded)

# Label encoding — assigns each category an integer (use only when order is meaningful,
# e.g., "Low"/"Medium"/"High" — NOT for unordered categories like department names)
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df_cat["department_encoded"] = encoder.fit_transform(df_cat["department"])
print(df_cat)
```

**Discussion prompt:** *"Why would label-encoding 'department' as Sales=0, HR=1, Engineering=2 be misleading to a model, compared to one-hot encoding?"* (Answer: it implies a false order/ranking — that Engineering > HR > Sales numerically — which isn't true for unordered categories.)

### 5.2 Scaling Numeric Variables

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler

data = pd.DataFrame({"age": [22, 25, 47, 35, 60], "income": [20000, 25000, 90000, 55000, 120000]})

# Standardization — mean 0, std 1 (uses Day 2's mean/std dev directly)
scaler = StandardScaler()
standardized = scaler.fit_transform(data)

# Normalization — scales everything to a 0-1 range
minmax_scaler = MinMaxScaler()
normalized = minmax_scaler.fit_transform(data)

print("Standardized:\n", standardized)
print("Normalized:\n", normalized)
```

> Point out: *"'age' ranges from 22-60, 'income' ranges from 20,000-120,000. Without scaling, many algorithms (like KNN on Day 11) would let 'income' completely dominate 'age' just because its numbers are bigger — not because it's actually more important."*

### 5.3 Log Transform for Skewed Data

```python
right_skewed = pd.Series(np.random.exponential(scale=2, size=500))

log_transformed = np.log1p(right_skewed)   # log1p = log(1 + x), safely handles 0s

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(right_skewed, bins=30)
axes[0].set_title("Before Log Transform")
axes[1].hist(log_transformed, bins=30)
axes[1].set_title("After Log Transform")
plt.show()
```

> This is the direct, practical follow-up to spotting skew visually on Day 4 — now students see the fix, not just the diagnosis.

### 5.4 Creating Simple New Features

```python
df_dates = pd.DataFrame({"signup_date": pd.to_datetime(["2023-01-15", "2023-06-20", "2024-02-10"])})

df_dates["signup_year"] = df_dates["signup_date"].dt.year
df_dates["signup_month"] = df_dates["signup_date"].dt.month
df_dates["days_since_signup"] = (pd.Timestamp.now() - df_dates["signup_date"]).dt.days

print(df_dates)
```

---

## 6. The Full EDA Workflow, End to End (15 minutes)

Run this live, on a slightly messy synthetic dataset, applying every step from Sections 2–5 in order:

```python
import pandas as pd
import numpy as np

np.random.seed(1)
df = pd.DataFrame({
    "age": list(np.random.randint(20, 60, 95)) + [np.nan]*5,
    "salary": list(np.random.normal(55000, 15000, 97)) + [np.nan, 500000, 600000],
    "department": np.random.choice(["Sales", "HR", "Engineering", np.nan], 100, p=[0.4, 0.3, 0.25, 0.05])
})

# Step 1-2: Structure & missing data
print(df.shape)
print(df.isnull().sum())

# Step 3: Duplicates
print(df.duplicated().sum())

# Step 4-5: Distributions & relationships (would normally plot here — Day 4 techniques)

# Step 6: Handle missing data
df["age"] = df["age"].fillna(df["age"].median())
df["salary"] = df["salary"].fillna(df["salary"].median())
df["department"] = df["department"].fillna(df["department"].mode()[0])

# Step 7: Handle outliers (IQR method on salary)
Q1, Q3 = df["salary"].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
df["salary"] = df["salary"].clip(upper=upper_bound)   # cap instead of removing

# Step 8: Feature engineering
df_encoded = pd.get_dummies(df, columns=["department"])

print(df_encoded.head())
print(df_encoded.isnull().sum())   # confirm nothing missing remains
```

---

## 7. Mini-Project 1 Brief (15 minutes)

### 7.1 Objective
Produce a complete EDA report on a real dataset of your choice, applying every technique from Module 1 (Days 1–5).

### 7.2 Choosing a Dataset
Pick any dataset you're genuinely curious about — options include:
- Kaggle datasets (https://www.kaggle.com/datasets) — pick anything with at least a few hundred rows and a mix of numeric + categorical columns
- A dataset from your own interests (sports stats, a hobby, a topic you follow)
- Any dataset already familiar from class demos, extended with your own questions

### 7.3 Deliverables
A GitHub repo containing:
1. A Jupyter/Colab notebook with:
   - Dataset overview (`.shape`, `.info()`, `.head()`)
   - Missing value analysis + how you handled it, and why
   - Duplicate check
   - At least 4 visualizations (mix of distribution and relationship plots from Day 4)
   - Outlier detection + your handling decision, and why
   - At least 2 engineered/encoded features
   - A short written summary (5–10 sentences) of what you found and what surprised you
2. A `README.md` describing the dataset, your goal, and your key findings

### 7.4 Rubric (What Gets Checked)

| Criterion | What we're looking for |
|---|---|
| Completeness | Every step in Section 6's EDA workflow is present |
| Correct technique choice | Sensible imputation/outlier decisions with reasoning given, not just default choices |
| Visualization quality | Plots are labeled, relevant, and actually support a conclusion |
| Written reasoning | Explains *why*, not just *what* — e.g., "used median because salary was right-skewed" |
| GitHub hygiene | Clear README, clean commit history, notebook runs top-to-bottom without errors |

### 7.5 Timeline
Due before Day 8 (start of Multiple/Polynomial Regression) — use time between sessions, plus any in-class work time, to complete it. A brief peer show-and-tell happens at the start of Day 8.

---

## 8. Recap (Last 5 minutes)

Quick verbal quiz (cold-call or hands-up):
1. Name the 8 steps of the EDA loop from Section 1, in order.
2. When would you choose median imputation over mean imputation?
3. What's the difference between the IQR method and the z-score method for detecting outliers?
4. Why can label-encoding an unordered category (like department names) mislead a model?
5. What is Mini-Project 1 due, and what does the rubric emphasize most?